In [7]:
# package imports
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, recall_score

from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
import sys
from pathlib import Path
from datasets import load_dataset


# manipulating the root path
sys.path.append(str(Path.cwd().parent))

from app.ml.preprocess import clean_text
# data loading and processing

dataset = load_dataset('stanfordnlp/imdb')

train_df = dataset['train'].to_pandas()
test_df = dataset['test'].to_pandas()

train_df['clean_text'] = train_df['text'].apply(clean_text)
test_df['clean_text'] = test_df['text'].apply(clean_text)

In [8]:
model_lr = Pipeline([
    (
        'tfidf', TfidfVectorizer(
            max_features=50000,
            ngram_range=(1,2)
        )),
        (
            'classifier', LogisticRegression()
        )
])

In [9]:
model_lr = Pipeline([
    (
        'tfidf', TfidfVectorizer(
            max_features=50000,
            ngram_range=(1,2)
        )),
        (
            'classifier', LinearSVC()
        )
])

In [10]:
import mlflow
import mlflow.sklearn

mlflow.set_experiment("imdb-sentiment")

with mlflow.start_run(run_name="Logistic Regression"):

    mlflow.log_param("model", "logistic_regression")
    mlflow.log_param("max_features", 50000)
    mlflow.log_param("ngram_range", "(1,2)")

    model_lr.fit(
        train_df["clean_text"],
        train_df["label"]
    )

    predictions = model_lr.predict(
        test_df["clean_text"]
    )

    accuracy_lr = accuracy_score(
        test_df["label"],
        predictions
    )

    f1_lr = f1_score(test_df['label'], predictions)
    recall_lr = recall_score(test_df['label'], predictions)


    mlflow.log_metric("accuracy", accuracy_lr)
    mlflow.log_metric("f1_score", f1_lr)
    mlflow.log_metric('recall', recall_lr)

    mlflow.sklearn.log_model(
        model_lr,
        "sentiment_model"
    )

2026/10/01 15:03:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/01 15:04:52 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


## Saving the existing model
saving this model for real world use

In [13]:
import joblib

joblib.dump(
    model_lr,
    "../models/sentiment_pipeline.pkl"
)

['../models/sentiment_pipeline.pkl']

## Testing the Prediction

In [14]:
model = joblib.load('../models/sentiment_pipeline.pkl')

review = 'This movie was absolutely fantastic!'

prediction = model.predict([review])

print(prediction)

[1]
